# Решения · ML Handbook гл. 43 (embeddings)

Открывай **только после** [handbook_gl43_практика.ipynb](./handbook_gl43_практика.ipynb).

Источник: [representation-learning.html](https://mlhandbook-production.up.railway.app/parts/deep-learning/representation-learning.html)

Теория: [../d.5.43 Обучение представлений и metric learning.md](../d.5.43 Обучение представлений и metric learning.md) · [README.md](./README.md)

## Зачем практику handbook

Закрепить главу **руками**: перед каждым блоком — **Зачем этот фрагмент** и ссылка на конспект; после запуска — **Что проверить** по `print`.

| ☐ | Блок |
|---|------|
| ☐ | §43.1 Embedding |
| ☐ | §43.2 triplet smoke |


## Импорты

Один раз в начале.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# §43.1 · nn.Embedding

## Зачем этот фрагмент

Таблица весов по id: слово, user, item. Основа recsys/NLP до трансформеров и внутри них (embedding слоя).

См. [d.5.43 §2](../d.5.43 Обучение представлений и metric learning.md#2-эмбеддинг-handbook-431).

## Что проверить

Три индекса → тензор **(3, 16)**.


In [ ]:
emb = nn.Embedding(1000, 16)
print(emb(torch.tensor([5, 42, 7])).shape)


---


# §43.2 · Triplet smoke

## Зачем этот фрагмент

Короткий цикл с `triplet_margin_loss`: anchor ближе к positive, чем к negative. Полная домашка handbook — t-SNE и hard negatives.

См. [d.5.43 §3](../d.5.43 Обучение представлений и metric learning.md#3-metric-learning-handbook-432).

## Что проверить

Loss считается и падает хотя бы немного; в print — конечное значение.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import numpy as np

d = load_digits()
X = torch.tensor((d.data / 16.0).astype('float32'))
y = torch.tensor(d.target)
Xtr, _, ytr, _ = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
enc = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 16))
opt = torch.optim.Adam(enc.parameters(), lr=1e-3)
rng = np.random.default_rng(0)
for _ in range(50):
    cls = int(rng.integers(0, 10))
    same = np.where(ytr.numpy() == cls)[0]
    diff = np.where(ytr.numpy() != cls)[0]
    ai = int(rng.choice(same)); pi = int(rng.choice(same)); ni = int(rng.choice(diff))
    loss = F.triplet_margin_loss(enc(Xtr[ai]), enc(Xtr[pi]), enc(Xtr[ni]), margin=0.5)
    opt.zero_grad(); loss.backward(); opt.step()
print('last triplet loss:', round(loss.item(), 4))
